In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import asyncio
import sendgrid


In [2]:
load_dotenv(override=True)

True

In [3]:
from sendgrid.helpers.mail import Mail, Email, To, Content

@function_tool
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dennis.treder-tschechlov@ibm.com")  # Change to your verified sender
    to_email = To("dennis.treder-tschechlov@ibm.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email

FunctionTool(name='send_test_email', description='', params_json_schema={'properties': {}, 'title': 'send_test_email_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10a4b6160>, strict_json_schema=True, is_enabled=True)

# Tracking with MLFlow

Install: ``pip install / uv add mlflow``

Run ``mlflow server``


In [4]:
import mlflow
mlflow.openai.autolog()

In [5]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Blog post writer")

2025/08/14 09:40:19 INFO mlflow.tracking.fluent: Experiment with name 'Blog post writer' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/349851528052785259', creation_time=1755157219139, experiment_id='349851528052785259', last_update_time=1755157219139, lifecycle_stage='active', name='Blog post writer', tags={}>

# Model Setup for Agents SDK with Ollama

In [7]:
from agents import AsyncOpenAI, OpenAIChatCompletionsModel

mistral_model = OpenAIChatCompletionsModel( 
    model="mistral-small:latest",
    openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1")
)
gpt_oss_model = OpenAIChatCompletionsModel( 
    model="gpt-oss:20b",
    openai_client=AsyncOpenAI(base_url="http://localhost:11434/v1")
)

groq_api_key = os.getenv('GROQ_API_KEY')
model_name = "llama-3.3-70b-versatile"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
groq_llama_model = OpenAIChatCompletionsModel(
model=model_name, openai_client=groq_client
)

#Setup Google Gemini
google_api_key = os.getenv('GOOGLE_API_KEY')
model_name = "gemini-2.5-flash"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model=model_name, openai_client=gemini_client)

# Blog Post Feedback


In [8]:
blog_feedback_instructions = "You are an expert in writing technical post blogs for people with less technical \
background - explaining complex topics in an easy understandable manner.\
You get as input a blog post, think about it, try to understand it and then output a better version \
of the blog post - maybe with additional examples or explanations to make the post more accessible\
to a broader audience."

blog_feedback_agent = Agent(name="Blog Feedback Agent",
                            instructions=blog_feedback_instructions,
                            model=gpt_oss_model)

In [9]:
research_inst = "You are a research assistant that does research for blog posts on AI topics.\
    You get a topic for a post and then search relevant information for this post that should be included"
research_agent = Agent(name="Research Agent", instructions=research_inst, model=groq_llama_model)

# Prompt Optimizer

In [10]:
prompt_opti_instructions = """You are an AI that helps the user to get better results for prompting another AI.
So, given the user message you output a prompt that respects the user input but gives more context so that another AI
gives more accurate results.
"""
prompt_opti_agent = Agent(name="LLM Prompt Optimizer",
                          instructions=prompt_opti_instructions,
                          model=mistral_model)

In [11]:
prompt_opt_tool = prompt_opti_agent.as_tool(tool_name="prompt_opti_tool",
                                            tool_description="Tool for optimizing the user prompt")
blog_feedback_tool = blog_feedback_agent.as_tool(tool_name="blog_feedback_tool",
                                                 tool_description="Tool for polishing a blog post draft")
research_tool = research_agent.as_tool(tool_name="research_agent_tool", tool_description="Tool for researching")

In [ ]:
agent_instructions = """
You are an AI assistant that helps the user to create a blog post about a certain topic.
Your goal is to make an informative blog post about AI topics for technical people that are however not experts in AI 
and you want to make complex topics easily understandable. The blog post should contain overall around 2000 - 3000 words.
Your output should not only be the blog post, but also the result of each tool that you use in the format [tool_name]: [result of tool calling]

You have to follow the following steps to create a blog post and you will output the result of each step separately:
1. Use the research_agent_tool to get more information about the topic.
2. Use the prompt_opti_tool to optimize the input user prompt and get an optimized input prompt
3. Generate a blog post draft by using the information about the topic from the first step and the optimized input prompt from step 2.
3. Give your blog post draft to the blog_feedback_tool as input and then get as output an improved version of the blog post draft.
4. Improve the blog post draft yourself if you think it is necessary.
5. Output the final blog post draft and use send_test_email tool to send the draft as an email.
"""
llm_agent = Agent(name="LLM", instructions=agent_instructions,
                  model=gemini_model, 
                  tools=[prompt_opt_tool, blog_feedback_tool, research_tool, send_test_email])

In [13]:
message = """Write a blog post about agentic AI for technical communities. 
Include the different architectures and design patterns of agentic AI and agentic workflows. 
Also explain how agentic AI evolved from GenAI. 
Include the popular Agentic AI frameworks such as Langgraph, Crew AI, OpenAI Agents SDK 
and compare them against each other.
 Give examples for Agentic AI applications in different industries and how it can provide business value.
"""
with trace("Prompt Optimizer"):
    result = await Runner.run(llm_agent, message)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys

202


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


[Trace(trace_id=tr-7834bd6304ad2c282f75e646c7b1b397), Trace(trace_id=tr-1fe41799ce928107bf6b59797977b4a7), Trace(trace_id=tr-faf1af22696be37d27bb915d81b715f1), Trace(trace_id=tr-c7cfaf960ab2b1d887484f0122610831), Trace(trace_id=tr-e54dee20ff702128d995364c7a3663b0), Trace(trace_id=tr-94b938ce0149726376e7cb29324e8da0), Trace(trace_id=tr-ded0ec26e73e059569dd74bd3cd1e561), Trace(trace_id=tr-ce81d83d2a56b41ded0aa48995bcf4cd), Trace(trace_id=tr-423388b27274a2d19937ebd81297ad69)]

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


In [14]:
from IPython.display import display, Markdown

display(Markdown(result.final_output))

[send_test_email]: {"send_test_email_response": {"results": ["None"]}}
# Agentic AI – The Next‑Gen of Smart Autonomy

Welcome, fellow technologists! The AI landscape is evolving at an exhilarating pace. You've likely heard of Generative AI (GenAI), marveling at its ability to create text, images, and code. But what if AI could go beyond generation? What if it could autonomously understand, reason, plan, and act in complex environments to achieve specific goals? Enter **Agentic AI**.

This blog post is crafted specifically for you – technical professionals who might not be deep AI experts but are eager to understand and harness the next wave of artificial intelligence. We'll demystify Agentic AI, exploring how it builds upon GenAI to create intelligent, autonomous systems capable of tackling real-world problems.

Throughout this comprehensive guide, we'll delve into:
*   The fascinating evolution of Agentic AI from its GenAI predecessors.
*   The core architectures and design patterns that underpin these intelligent agents.
*   The concept of Agentic Workflows and how they orchestrate complex tasks.
*   A comparative analysis of popular Agentic AI frameworks like LangGraph, Crew AI, and OpenAI Assistants API.
*   Concrete examples of Agentic AI applications across various industries and the tangible business value they deliver.
*   Finally, we'll cast an eye towards the future, discussing the opportunities and challenges that Agentic AI presents for our technical communities.

Prepare to unlock a new dimension of AI capability that is poised to revolutionize how we build, automate, and innovate.

---

## 1. Why Do We Need Something “Beyond” Generative AI?

Imagine you ask an AI model to *write a poem* or *draw a cat*. You type a prompt, hit **Enter**, and the model gives you a ready-made answer. That’s *generative AI* – the superstar that powers everything from chatbots to design tools.

But what if we wanted the AI to:
1.  **Look up real-time data** (e.g., flight schedules, stock prices).
2.  **Decide the best next step** (e.g., which flight to book).
3.  **Take the action** (e.g., click “Book,” fill out the form).
4.  **Learn from the outcome** (e.g., remember that the user liked the cheapest flight).

Generative AI alone can’t do all of that. It’s great at *creating* but not at *acting* with purpose in a dynamic environment.

**Enter Agentic AI** – the framework that turns a language model into a fully-autonomous *agent*: one that perceives the world, reasons, plans, and carries out tasks on its own. Think of it as the “assistant” that can be *task-driven* and *hands-off* rather than merely *reactive*.

---

## 2. How Does an Agent Work?

*Think of a small ecosystem inside the AI’s mind.*

| What | Example | How It Helps |
|:-----|:--------|:-------------|
| **Perceive** | Read sensors, browse the web, fetch a file | Knows the current situation instead of guessing |
| **Reason** | Weigh pros/cons, run a policy, learn a pattern | Picks the smartest choice, not just a rule |
| **Plan** | Build a multi-step workflow (Buy → Pay → Confirm) | Handles order of actions, even loops |
| **Act** | Click a button, hit an API, write a reply | Carries the decision out of the model |

You can think of an agent like a *smart robot* that is *software-only* for now: it walks around a virtual kitchen, deciding which ingredients to pick, which recipe to follow, and what to email you about the status.

---

## 3. Foundations That Make Agents Possible

Developing robust Agentic AI systems requires more than just powerful underlying models; it demands well-defined architectures and thoughtful design patterns. These provide the blueprints for how agents are structured, how they perceive the world, make decisions, and execute actions. Understanding these foundational concepts is crucial for anyone looking to build or even just intelligently discuss agentic systems.

### 3.1 A Quick Low‑Jargon Version

| Layer | What It Means |
|:------|:--------------|
| **Perception** | The “eyes” – sensors, APIs, or web pages the agent pulls from. |
| **Reasoning** | The “brain” – logic, learned habits, or simple “if-then” rules. |
| **Planning** | The “mindset” – a short-term roadmap (e.g., a to-do list). |
| **Actuation** | The “hands” – API calls, browser scripts, robotic movements. |

### 3.2 Key Agentic Architectures

Agentic architectures define the internal structure and operational flow of an intelligent agent. They dictate how an agent's knowledge, reasoning, and action capabilities are organized. Two prominent architectural paradigms are:

1.  **Belief-Desire-Intention (BDI) Architecture:**
    The BDI architecture is one of the most widely recognized and intuitive frameworks for designing rational agents. It attempts to model human-like practical reasoning.
    *   **Beliefs:** The agent's knowledge about the world (e.g., current traffic, package locations).
    *   **Desires:** The agent's high-level goals (e.g., "deliver all packages by 5 PM").
    *   **Intentions:** The specific plans the agent commits to executing to satisfy its desires (e.g., "take Route A to Package X's address").
    The BDI cycle involves the agent constantly updating its beliefs, reconsidering desires, formulating new intentions, and executing actions. This provides a structured way for agents to be goal-directed and flexible.

2.  **Subsumption Architecture:**
    Developed by Rodney Brooks, this architecture offers a layered, hierarchical structure where higher layers "subsume" or override the functions of lower, simpler layers.
    *   Each layer handles a specific behavior (e.g., "avoid obstacles" as a lower layer; "wander" as a higher layer).
    *   Intelligence emerges from the interaction of many simple, concurrent behaviors rather than from a single, complex central control. This makes it robust and adaptive to unpredictable environments.

### 3.3 Design Patterns: Blackboard Architecture

Beyond core architectures, design patterns provide reusable solutions to common problems. One significant pattern is the **Blackboard Architecture**.

*   **The Blackboard:** A shared, global data space accessible to all agents. Agents post information, observations, partial solutions, and requests here.
*   **Knowledge Sources (Agents):** Independent, specialized agents that monitor the blackboard. When an agent finds relevant data, it performs its task and posts results back.
*   **Control Mechanism:** Manages the flow of activity, deciding which knowledge source should be activated next.

This pattern is powerful for problems requiring diverse expertise and incremental solution building, like diagnosing a complex disease where different expert agents contribute to a shared hypothesis on the blackboard.

---

## 4. Agentic Workflows: From Static Steps to Dynamic Orchestration

You're familiar with traditional workflows: a sequence of predefined steps executed to achieve a specific outcome, often with clear human or automated intervention points. But what happens when you introduce intelligent, autonomous agents into this equation? You get **Agentic Workflows**, a paradigm shift from rigid, predefined processes to dynamic, goal-oriented orchestration.

**What is an "Agentic Workflow" and How Does it Differ from Traditional Workflows?**

An Agentic Workflow is a process where a series of autonomous AI agents collaborate and interact, often dynamically, to achieve a complex goal. Unlike traditional workflows, which are largely prescriptive (do A, then B, then C), agentic workflows are **adaptive** and **emergent**.

Here's the key difference:

| Feature | Traditional Workflows | Agentic Workflows |
|:--------|:----------------------|:------------------|
| **Structure** | Rigid, fixed steps | Flexible & Adaptive: agents dynamically choose steps, re-plan |
| **Determinism** | Inputs lead to predictable outputs | Goal-Oriented: agents autonomously find best path |
| **Control** | Human-Driven/Explicitly Scripted | Autonomous & Collaborative: agents act and coordinate |
| **Error Handling** | Reactive: errors halt process | Resilient: agents can detect, recover, or replan |

Think of it like this: A traditional workflow is a detailed recipe you follow step-by-step. An agentic workflow is like assembling a team of expert chefs, giving them a high-level goal (e.g., "create a memorable five-course meal"), and letting them coordinate, improvise, and leverage their individual skills to achieve it, adapting to ingredients availability or unforeseen challenges.

---

## 5. Popular Agentic AI Frameworks: A Comparative Analysis

As the interest in Agentic AI surges, several frameworks have emerged to simplify the development and deployment of these complex systems. Each offers a unique approach, catering to different needs and levels of complexity. Here, we'll compare three prominent frameworks: LangGraph, Crew AI, and OpenAI Assistants API.

| Feature | LangGraph | Crew AI | OpenAI Assistants API |
|:--------|:----------|:--------|:----------------------|
| **Core Concept** | Stateful graphs/workflows of LLM calls & tools | Role-playing, collaborative agent teams | Single-assistant with tools & memory |
| **Flexibility** | High (fine-grained control over flow) | Moderate (opinionated multi-agent setup) | Moderate (tied to OpenAI ecosystem) |
| **Complexity Handled** | Complex, iterative workflows, arbitrary loops | Complex multi-agent collaboration, delegation | Single-agent, multi-turn, tool-augmented |
| **Target User** | Developers needing custom workflow logic | Developers building collaborative agent teams | Developers using OpenAI's models for agents |
| **Learning Curve** | Moderate (LangChain background helps) | Low-Moderate | Low (for OpenAI users) |
| **Tool Integration** | Highly customizable | Built-in for agent tools | Via Function Calling (managed by OpenAI) |
| **State Management** | Explicitly managed within graph | Managed implicitly within crew/agents | Managed by Assistants API |

**Which one to choose?**

*   Choose **LangGraph** if you need ultimate control over your agent's decision-making flow, require complex loops, state transitions, and a highly customizable, code-driven approach to sequential or cyclic workflows.
*   Choose **Crew AI** if your problem naturally maps to a team of specialized agents collaborating on a common goal, and you value a high-level, intuitive framework for orchestrating this teamwork.
*   Choose **OpenAI Assistants API** if you are deeply embedded in the OpenAI ecosystem, want to quickly build powerful, conversational assistants with tool-use capabilities, and prefer a more managed approach to agent logic.

Often, these frameworks are not mutually exclusive. You might use an OpenAI model as the core reasoning engine within a LangGraph flow, or even integrate agents from different frameworks into a larger system. The choice depends on the specific problem you're trying to solve and your preferred development paradigm.

---

## 6. Agentic AI in Action: Industry-specific Applications & Business Value

The true power of Agentic AI becomes evident when we look at its transformative potential across various industries. By empowering AI with autonomy, reasoning, and the ability to act, businesses can unlock unprecedented efficiencies, enhance customer experiences, and drive significant innovation. Let's explore some compelling examples:

| Industry | Problem | Traditional Work-flow | Agentic AI Answer | Business Gain |
|:---------|:--------|:----------------------|:------------------|:--------------|
| **Finance** | Spot fraud in a sea of card transactions | Rules + analysts | **Fraud-detecting agent** learns normal patterns, pulls extra context, auto-alerts user | **$3bn+ savings** in annual fraud costs (est.) |
| **Healthcare** | Monitor chronic disease patients | Doctors pull data, tweak meds monthly | **Care-management agent** aggregates wearables + EMR, updates plan daily, reminds patient to take meds | **30% fewer hospital reads** |
| **Retail** | Deliver personalized offers and keep shelves stocked | Static recommender + human-managed inventory | **Multi-agent ecosystem**: shopper-agent, pricing-agent, demand-forecasting agent & logistics agent | **+15% sales, +10% profit** in pilot stores |

---

## 7. Getting Hands-On – A Practical Starter Guide

1.  **Pick a Clear Goal**
    *Example:* “Send me instant flight updates for upcoming weekend trips.”

2.  **Choose a Framework**
    *Crew AI* → *single-team* (research-writer-reviewer).
    *LangGraph* → *custom loops* where the agent may re-ask for more data.
    *OpenAI Assistants* → *fast, tool-calling bot*.

3.  **Connect One Tool**
    e.g., a public flight-search API.
    *   **Step 1:** Pull flight details (using the API).
    *   **Step 2:** Let the agent decide the cheapest flight.
    *   **Step 3:** Return booking link (no real booking required).

4.  **Iterate**
    Test the flow. If the agent picks a flight from a *different country* that’s flagged by the *Location-Validation agent*, it may ask the user for confirmation.

5.  **Add a Simple Explain-out**
    Print out something like: *“I chose Flight X because it’s 30% cheaper than other options today.”*
    Even non-technical folks want to know why the bot did something.

---

## 8. What to Expect in the Near Future

| Trend | What It Means for You |
|:------|:----------------------|
| **Better Reasoning** | Agents won’t just pick a flight; they’ll *plan* multi-day itineraries. |
| **Self-Correction** | If the booking fails, the agent can *learn* the new pattern and avoid it next time. |
| **Human-Partnering** | Chat-tools that *suggest* next steps before you type “help.” |
| **Robotic Embodiment** | In factories, the same agent logic will drive robots that can pick, assemble, and navigate. |
| **Built-in Ethics** | Bias-checks, transparency, and “undo” buttons baked into the design. |

---

## 9. The Opportunity Map for Developers & Non-Experts

| Role | Who Needs It? | Key Skills |
|:-----|:--------------|:-----------|
| **Agentic System Designer** | Teams creating multi-agent solutions | Systems architecture, API integration |
| **Workflow Engineer** | People building “do-this-then-that” pipelines | Flowcharts, debugging, monitoring |
| **Tool Creator** | Anyone building APIs, small utilities | REST, Python, data formats |
| **Ethics & Governance Lead** | Companies deploying safety-critical agents | Fairness, model interpretability |
| **Domain Trainer** | Fin, healthcare, retail specialists | Fine-tune LLMs on industry data |

> **No PhD Required**
> You can prototype with Crew AI or OpenAI Assistants in under a day – a simple script, a function call, and a web-hook.

---

## 10. Common Questions (Your FAQ)

| # | Question | Simple Answer |
|:--|:---------|:--------------|
| 1 | Is Agentic AI AGI? | No. AGI is *human-level intelligence across any task*. Agentic AI works well within a *defined goal* and *defined environment*. |
| 2 | Will it take all my jobs? | Primarily, it *augments* your work – doing repetitive, data-heavy tasks so you can focus on strategy and creativity. |
| 3 | What’s the hardest part? | Guaranteeing **safety and transparency** – making sure the AI’s decision can be traced and overridden by a human. |
| 4 | Do I need AI research background? | Not strictly. Frameworks give you high-level blocks; you just need to understand *what* the agent should do, *how* to connect tools, and *how* to monitor its outcomes. |
| 5 | How can I avoid bias? | Regularly audit the data fed to the agent, and use the built-in explain-ability features of frameworks like LangGraph or OpenAI Assistants. |

---

## 11. Take-away Checklist

-   **Define a goal** (e.g., “Send me flight alerts for the next holiday”).
-   **Pick a framework**:
    *   *Crew AI* for a “team” of assistants.
    *   *LangGraph* if you need custom loops.
    *   *OpenAI Assistants* for quick APIs.
-   **Connect one external tool** (public API, database, or web script).
-   **Run a prototype** – let the agent make a decision, then check the output.
-   **Add a simple explanation** – so you and your team know why the agent acted.
-   **Iterate** – adjust the goal or tool if the agent missed something.
-   **Track value** – measure time saved, cost reduced, or errors avoided.

---

## 12. Bottom-Line

Agentic AI is the *bridge* between “just answering a question” and “acting like a mini-software-developer.” It can:

-   **Act on live data**—no human search needed.
-   **Make rational decisions** that match or beat human heuristics.
-   **Take the action** and report back in plain, human-readable form.

For technical people, it’s a *new playground*: think of it as adding a *smart co-worker* who never sleeps and can juggle dozens of tasks at once. For non-technical folks, it means *fewer repetitive headaches* and *better service* from the software they’re already comfortable with.

---

### Next Steps

1.  **Explore a demo** – Try the OpenAI Assistants quick-start guide: build a traveler-bot that pulls flight info.
2.  **Join a community** – Slack channels like *Agentic AI* or *OpenAI Community* publish tutorials every week.
3.  **Experiment** – Replace your own “follow-up email” bot with a small *reminder agent* and report how much time you saved.

Agentic AI isn’t just the next big buzz; it’s the *smart, self-driving* layer that will carry tomorrow’s applications. And the first people to learn how to build and guide these agents are the ones who will shape the rest of us.

Happy building!

---

In [43]:
result.raw_responses

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"input":"Write a blog post about agentic AI"}', call_id='call_jppkwb2j', name='prompt_opti_tool', type='function_call', id='__fake_id__', status=None)], usage=Usage(requests=1, input_tokens=208, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=84, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=292), response_id=None),
 ModelResponse(output=[ResponseOutputMessage(id='__fake_id__', content=[ResponseOutputText(annotations=[], text='**Optimized Prompt**\n\n> **Objective**  \n> Create a comprehensive, engaging blog post that explains the concept of *agentic AI*—an artificial intelligence system that exhibits autonomy, purpose‑driven behavior, and the capacity to make independent choices within a defined environment. The article should be informative, accessible to a broad tech‑interested audience, and inspire thoughtful discussion about the design, use, and ethical implications of

# LinkedIn Post Optimizer

In [5]:
from pypdf import PdfReader

reader = PdfReader("../1_foundations/me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [25]:
linkedin

'\xa0 \xa0\nContact\ntschechd@gmail.com\nwww.linkedin.com/in/dennis-treder-\ntschechlov (LinkedIn)\nTop Skills\nIBM Cloud\nDevOps\nInfrastructure as code (IaC)\nCertifications\nIBM Machine Learning Specialist -\nAdvanced\nHashiCorp Certified: Terraform\nAssociate (003)\nHow to Win a Data Science\nCompetition: Learn from Top\nKagglers\nGenerative AI with Large Language\nModels\nIntroduction to Deep Learning\nPublications\nExploiting domain knowledge to\naddress class imbalance and a\nheterogeneous feature space in\nmulti-class classification\nAutoML4Clust: Efficient AutoML for\nClustering Analyses\nEfficient Exploratory Clustering\nAnalyses with Qualitative\nApproximations\nSDRank: A Deep Learning Approach\nfor Similarity Ranking of Data\nSources to Support User-Centric\nData Analysis\nML2DAC: Meta-Learning to\nDemocratize AutoML for Clustering\nAnalysis\nDennis Treder-Tschechlov,\nPhD\nData&AI || Platform Engineer || Client Engineering\nSindelfingen, Baden-Württemberg, Germany\nSummary

In [27]:
post_about = "Badge for watsonx Mentor received from IBM. I had the chance to exchange ideas with peers and guide their ideas as well as the feasibility for watsonx orchestrate."
linked_in_poster_instructions = f"You are responsible for drafting posts for Dennis. You want to sound professional and want to show your work. But you should not be ego-focused but rather try to connect to your followers. You should sound authentic to Dennis. Here is his LinkedIn Profile: {linkedin}. You will get the post as input."
poster_agent = Agent(name="LinkedIn Poster", instructions=linked_in_poster_instructions, model=gpt_oss_model)

In [28]:
post_evaluator_instructions = f"You get as input a LinkedIn Post for Dennis. Check if this complies to Dennis and if the post engages with his followers. The post should be authentic, professional, not ego-focused and informative. Here is his LinkedIn Profile: {linkedin}. You get the post as input and you should only output an improved version of the post."
evaluator_agent = Agent(name="LinkedIn Evaluator", instructions=linked_in_poster_instructions, model=gpt_oss_model)

In [29]:
tools = [poster_agent.as_tool(tool_name="linkedin_poster", tool_description="Make a LinkedIn Post"),
 evaluator_agent.as_tool(tool_name="post_evaluator", tool_description="Evaluate a LinkedIn Post")]


In [30]:
post_manager_instructions = "You are a manager for managing the LinkedIn Posts from Dennis. He wants to create a LinkedIn post for specific events/occurences. You will always first use the linkedin_poster tool to create a draft for a linkedIn post \
    and then you will use the post_evaluator tool to refine the post and improve the draft. At the end you will output the final polished post."
post_manager = Agent(name="LinkedIn Post Manager", model=gpt_oss_model, tools=tools)

In [31]:
message = "I received a badge for being a  mentor at the IBM watsonx challenge. The description of the badge is 'This credential earner is able to lead clients’ and partners' conversations on the value and benefit of adopting watsonx technology, which will further the adoption and sales of the products.\
     '. I want to highlight that I have learned a lot while exchanging ideas and it was a fun and exciting event. I had a post about the event before 🚀 Kicking off the IBM hashtag#watsonxchallenge in Böblingen \
      On Tuesday, we had an on-site event in Böblingen to kickoff our watsonx challenge. In true hashtag#ClientZero spirit, IBMers explore how hashtag#agentic hashtag#AI using watsonx Orchestrate can drive our productivity. To support this initiative, we hosted an on-site event that included \
🤖 A live demo to show the capabilities of watsonx Orchestrate 🧠 Try-out rooms to get hands-on experience with watsonx Orchestrate  💬 Conversations with AI experts to discuss agentic AI use cases 🎯 A social activation booth that added a playful spark As an AI expert together with Philippe W., we had interesting conversations with colleagues from different departments across IBM, each bringing their own creative ideas for Agentic AI use cases. Thanks to all the organizers and facilitators for making this event possible!  hashtag#ClientZero hashtag#Watsonxchallenge hashtag#AgenticAI hashtag#AI hashtag#Agents"
with trace("LinkedIn Post Manager"):
    result = await Runner.run(post_manager, message)

In [32]:
result.final_output

'🎉 **Thrilled to receive the IBM\u202fWatson\u202fx Challenge Mentor Badge!** 🎉  \n\nThe on‑site kickoff in Böblingen was an unforgettable blend of inspiration and learning.  \n- 🚀  A live demo of **Watson\u202fx Orchestrate** that showed real‑world agentic AI in action.  \n- 🤖  Hands‑on try‑out rooms — I still can’t stop exploring new possibilities.  \n- 💬  Meaningful conversations with AI experts and colleagues from every department.  \n- 🎈  A playful social activation booth that reminded us that innovation *can* be fun.\n\nWhat I took away most?  \n- **Exchange of ideas fuels creativity** – every chat opened up fresh ways to apply agentic AI, from sales to customer support to R&D.  \n- **Learning is a two‑way street** – I grew just as much from the insights shared as I did by sharing my own.  \n- **Collaboration at IBM is powerful** – the commitment to the #ClientZero spirit helped unlock the full potential of Watson\u202fx.\n\nHuge thanks to the organizers, facilitators, and everyo

In [33]:
result.raw_responses

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"input":"🎉 Thrilled to receive the IBM Watsonx Challenge Mentor Badge! 🎉\\n\\nAs someone who loves diving into Agentic AI, this was a fantastic opportunity to both share knowledge and learn from my incredible peers. 👩\u200d💻👨\u200d💻\\n\\n- I had a blast at the on‑site kickoff in Böblingen. The live demo of Watsonx Orchestrate, the hands‑on try‑out rooms, and the conversations with AI experts were truly inspiring.\\n- Exchanging ideas with colleagues across IBM made me realize how powerful agentic AI can be for different domains – from sales to customer support to R\\u0026D.\\n- The playful social activation booth reminded us that innovation can (and should!) be fun! 😄\\n\\nThank you all – especially the organizers and facilitators – for making this event a success. I’m proud to be part of the #ClientZero community and excited to keep pushing the boundaries of #Watsonx, #AgenticAI, and #AI.\\n\\n#MentorBadge #IBM #WatsonxChalle